# BraTS 2023-Style nnU-Net v2 Pipeline from Google Drive (T1c Only)

This notebook rebuilds the pipeline around **T1c only** training and upgrades it toward stronger BraTS-style scoring while keeping the run inside a Colab Pro budget.

## Budget defaults in this version

- Trains **one fold only** by default (`fold 0`). This avoids the accidental 5-fold cost explosion.
- Defaults to **100 epochs** instead of 250. Based on your observed A100 timing of about 80 hours for 250 epochs, this targets about 32 hours for one fold, or about 172 compute units at 5.37 CU/hour.
- Saves `checkpoint_latest.pth` every epoch and `checkpoint_best.pth` whenever validation improves.
- Mirrors checkpoints into the local project folder under `checkpoints/` so stopping the training cell still leaves a resumable checkpoint outside the nnU-Net results tree.
- To intentionally train more folds, set `BRATS_ALLOW_MULTI_FOLD=1` and `BRATS_TRAIN_FOLDS=0,1,...`. Leave it off for budget safety.

## What changed

- Uses **only source channel `0001` -> exported `_0000` -> `T1c`**
- Trains nnU-Net on **BraTS region targets** instead of raw labels:
  - `whole_tumor (WT) = [1, 2, 3]`
  - `tumor_core (TC) = [2, 3]`
  - `enhancing_tumor (ET) = [3]`
- Evaluates with the **official BraTS-style regional metrics**:
  - Dice
  - HD95
- Uses the newer **nnU-Net ResEnc presets**, defaulting to **ResEnc M** for A100 budget control

## Sources

- nnU-Net docs: https://github.com/MIC-DKFZ/nnUNet
- ResEnc presets: https://github.com/MIC-DKFZ/nnUNet/blob/master/documentation/resenc_presets.md
- Region-based training: https://github.com/MIC-DKFZ/nnUNet/blob/master/documentation/region_based_training.md
- BraTS pages: https://www.med.upenn.edu/cbica/brats/
- BraTS 2021 data description: https://www.med.upenn.edu/cbica/brats2021/
- BraTS task/evaluation background: https://www.med.upenn.edu/cbica/brats2019/tasks.html and https://www.med.upenn.edu/cbica/brats2019/evaluation.html
- BraTS 2023 winning paper: https://arxiv.org/abs/2402.17317

## 1. Install dependencies

In [1]:
%pip install -q nibabel matplotlib scipy ipywidgets nnunetv2


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 205.6/205.6 kB 6.6 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 9.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.1/63.1 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.7/73.7 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.8/52.8 MB 50.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 26.5/26.5 MB 97.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 

## 2. Mount Drive, locate the dataset, and choose a budget-safe nnU-Net preset

In [17]:
from __future__ import annotations

import json
import os
import re
import shutil
import signal
import subprocess
from collections import Counter
from pathlib import Path

import numpy as np
import nibabel as nib

IN_COLAB = False
try:
    from google.colab import drive
    IN_COLAB = True
except ModuleNotFoundError:
    drive = None




def first_existing(paths: list[Path]) -> Path | None:
    for path in paths:
        if path.exists():
            return path
    return None


if IN_COLAB:
    drive.mount('/content/drive')
    PROJECT_ROOT = first_existing([
        Path('/content/gp2'),
        Path('/content/drive/MyDrive/gp2'),
        Path('/content/drive/MyDrive/GP2'),
        Path.cwd(),
    ]).resolve()
    candidate_data_roots = [
        Path('/content/drive/MyDrive/Brats Final/Processed BraTS 128'),
        Path('/content/drive/MyDrive/Brats_Final_Data/Processed BraTS 128'),
        Path('/content/drive/MyDrive/Brats_Final_Data/Processed_BraTS_128'),
        Path('/content/drive/MyDrive/Brats Final/Processed_BraTS_128'),
    ]
    WORKSPACE_DIR = Path('/content/brats_workspace')
    BACKUP_ROOT = Path('/content/drive/MyDrive/brats_full_backup_t1cV2')
else:
    PROJECT_ROOT = Path.cwd().resolve()
    candidate_data_roots = [
        Path('C:/Users/MSI/Downloads/Brats Final/Processed BraTS 128'),
        Path('C:/Users/MSI/Downloads/Brats_Final_Data/Processed BraTS 128'),
        Path('C:/Users/MSI/Downloads/Brats_Final_Data/Processed_BraTS_128'),
        Path('C:/Users/MSI/Downloads/Brats Final/Processed_BraTS_128'),
    ]
    WORKSPACE_DIR = PROJECT_ROOT / 'workspace_local_t1cV2'
    BACKUP_ROOT = PROJECT_ROOT / 'brats_full_backup_t1cV2'

DATA_ROOT = first_existing(candidate_data_roots)
if DATA_ROOT is None:
    raise FileNotFoundError(
        'Could not find the dataset root. Expected a Processed BraTS 128 folder in Drive or locally.'
    )

TRAIN_ROOT = DATA_ROOT / 'train'
EVAL_ROOT = first_existing([DATA_ROOT / 'val', DATA_ROOT / 'test'])
EVAL_SPLIT_NAME = 'val' if (DATA_ROOT / 'val').exists() else 'test'

if not TRAIN_ROOT.exists():
    raise FileNotFoundError(f'Missing train directory: {TRAIN_ROOT}')
if EVAL_ROOT is None:
    raise FileNotFoundError(f'Could not find val/ or test/ under: {DATA_ROOT}')


def choose_resenc_preset() -> tuple[str, float | None]:
    requested = os.environ.get('BRATS_RESENC_PRESET', 'M').strip().upper()
    if requested not in {'M', 'L', 'XL'}:
        raise ValueError('BRATS_RESENC_PRESET must be one of: M, L, XL')

    try:
        import torch
        if torch.cuda.is_available():
            memory_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
            return requested, memory_gb
    except Exception:
        pass
    return requested, None


def parse_train_folds() -> list[int]:
    requested_text = os.environ.get('BRATS_TRAIN_FOLDS', '0')
    requested = [int(fold.strip()) for fold in requested_text.split(',') if fold.strip()]
    if not requested:
        requested = [0]
    invalid = [fold for fold in requested if fold not in {0, 1, 2, 3, 4}]
    if invalid:
        raise ValueError(f'Invalid folds {invalid}; nnU-Net folds must be 0..4')

    allow_multi_fold = os.environ.get('BRATS_ALLOW_MULTI_FOLD', '0').strip() == '1'
    if len(requested) > 1 and not allow_multi_fold:
        print(
            'Budget guard: BRATS_TRAIN_FOLDS requested multiple folds, but '
            'BRATS_ALLOW_MULTI_FOLD is not 1. Keeping only the first fold:', requested[0]
        )
        requested = [requested[0]]
    return requested


RESENC_PRESET, GPU_MEMORY_GB = choose_resenc_preset()
PLANNER_NAME = f'nnUNetPlannerResEnc{RESENC_PRESET}'
PLANS_NAME = f'nnUNetResEncUNet{RESENC_PRESET}Plans'
MAX_EPOCHS = int(os.environ.get('BRATS_MAX_EPOCHS', '250'))
SAVE_EVERY_N_EPOCHS = int(os.environ.get('BRATS_SAVE_EVERY_N_EPOCHS', '1'))
TRAINER_NAME = f'nnUNetTrainerBudget{MAX_EPOCHS}Epochs'
CONFIGURATION = '3d_fullres'

DATASET_ID = 303
DATASET_NAME = 'ProcessedBraTS128_T1c_Regions'
DATASET_FOLDER = f'Dataset{DATASET_ID:03d}_{DATASET_NAME}'

NNUNET_RAW = WORKSPACE_DIR / 'nnUNet_raw'
NNUNET_PREPROCESSED = WORKSPACE_DIR / 'nnUNet_preprocessed'
NNUNET_RESULTS = WORKSPACE_DIR / 'nnUNet_results'
DATASET_ROOT = NNUNET_RAW / DATASET_FOLDER
IMAGES_TR = DATASET_ROOT / 'imagesTr'
LABELS_TR = DATASET_ROOT / 'labelsTr'
IMAGES_TS = DATASET_ROOT / 'imagesTs'

SOURCE_CHANNELS = [
    ('0000', 'T1c', 1),
]

TRAIN_FOLDS = parse_train_folds()
CHECKPOINT_MIRROR_DIR = Path(os.environ.get(
    'BRATS_CHECKPOINT_MIRROR',
    str(PROJECT_ROOT / 'checkpoints' / DATASET_FOLDER / f'{TRAINER_NAME}__{PLANS_NAME}__{CONFIGURATION}'),
)).resolve()

MODEL_BASE = (
    NNUNET_RESULTS
    / DATASET_FOLDER
    / f'{TRAINER_NAME}__{PLANS_NAME}__{CONFIGURATION}'
)

A100_COMPUTE_UNITS_PER_HOUR = float(os.environ.get('BRATS_A100_CU_PER_HOUR', '5.37'))
COLAB_COMPUTE_UNITS_LEFT = float(os.environ.get('BRATS_COMPUTE_UNITS_LEFT', '200'))
ESTIMATED_HOURS_PER_250_EPOCH_FOLD = float(os.environ.get('BRATS_HOURS_PER_250_EPOCH_FOLD', '80'))
ESTIMATED_HOURS_PER_FOLD = ESTIMATED_HOURS_PER_250_EPOCH_FOLD * (MAX_EPOCHS / 250)
ESTIMATED_TOTAL_HOURS = ESTIMATED_HOURS_PER_FOLD * len(TRAIN_FOLDS)
ESTIMATED_TOTAL_CU = ESTIMATED_TOTAL_HOURS * A100_COMPUTE_UNITS_PER_HOUR

print('Running in Colab:', IN_COLAB)
print('Project root       :', PROJECT_ROOT)
print('Data root          :', DATA_ROOT)
print('Train root         :', TRAIN_ROOT)
print('Eval split         :', EVAL_SPLIT_NAME)
print('Eval root          :', EVAL_ROOT)
print('Workspace          :', WORKSPACE_DIR)
print('Backup root        :', BACKUP_ROOT)
print('Dataset folder     :', DATASET_FOLDER)
print('Export channels    :', SOURCE_CHANNELS)
print('ResEnc preset      :', RESENC_PRESET)
print('Planner            :', PLANNER_NAME)
print('Plans              :', PLANS_NAME)
print('Trainer            :', TRAINER_NAME)
print('Train folds        :', TRAIN_FOLDS)
print('Max epochs         :', MAX_EPOCHS)
print('Save every epochs  :', SAVE_EVERY_N_EPOCHS)
print('Checkpoint mirror  :', CHECKPOINT_MIRROR_DIR)
print(f'Estimated training : {ESTIMATED_TOTAL_HOURS:.1f} hours, {ESTIMATED_TOTAL_CU:.1f} compute units')
if ESTIMATED_TOTAL_CU > COLAB_COMPUTE_UNITS_LEFT:
    print(
        f'Warning: estimated training cost exceeds your configured {COLAB_COMPUTE_UNITS_LEFT:.1f} CU budget. '
        'Lower BRATS_MAX_EPOCHS or keep only one fold.'
    )
if GPU_MEMORY_GB is not None:
    print(f'GPU memory (GB)    : {GPU_MEMORY_GB:.2f}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Running in Colab: True
Project root       : /content/drive/MyDrive/gp2
Data root          : /content/drive/MyDrive/Brats_Final_Data/Processed_BraTS_128
Train root         : /content/drive/MyDrive/Brats_Final_Data/Processed_BraTS_128/train
Eval split         : val
Eval root          : /content/drive/MyDrive/Brats_Final_Data/Processed_BraTS_128/val
Workspace          : /content/brats_workspace
Backup root        : /content/drive/MyDrive/brats_full_backup_t1cV2
Dataset folder     : Dataset303_ProcessedBraTS128_T1c_Regions
Export channels    : [('0000', 'T1c', 1)]
ResEnc preset      : M
Planner            : nnUNetPlannerResEncM
Plans              : nnUNetResEncUNetMPlans
Trainer            : nnUNetTrainerBudget250Epochs
Train folds        : [0]
Max epochs         : 250
Save every epochs  : 1
Checkpoint mirror  : /content/drive/MyDrive/gp2/checkpoints/Dataset303_P

## 3. Collect image/mask pairs from Drive

In [3]:
IMAGE_SUFFIX = '_image.nii.gz'
MASK_SUFFIX = '_mask.nii.gz'


def sanitize_case_id(text: str) -> str:
    value = re.sub(r'[^A-Za-z0-9_]+', '_', text.strip())
    value = re.sub(r'_+', '_', value).strip('_')
    if not value:
        raise ValueError(f'Invalid case id from {text!r}')
    return value


def collect_pairs(split_root: Path) -> list[dict]:
    rows = []
    for tumor_dir in sorted(path for path in split_root.iterdir() if path.is_dir()):
        images = {}
        masks = {}
        for file_path in sorted(tumor_dir.glob('*.nii.gz')):
            name = file_path.name
            if name.endswith(IMAGE_SUFFIX):
                images[name[:-len(IMAGE_SUFFIX)]] = file_path
            elif name.endswith(MASK_SUFFIX):
                masks[name[:-len(MASK_SUFFIX)]] = file_path

        common = sorted(set(images) & set(masks))
        missing_images = sorted(set(masks) - set(images))
        missing_masks = sorted(set(images) - set(masks))

        if missing_images:
            print(f'{tumor_dir.name}: masks without images -> {missing_images[:5]}')
        if missing_masks:
            print(f'{tumor_dir.name}: images without masks -> {missing_masks[:5]}')

        for stem in common:
            rows.append(
                {
                    'case_id': sanitize_case_id(f'{tumor_dir.name}_{stem}'),
                    'tumor_type': tumor_dir.name,
                    'stem': stem,
                    'image_path': images[stem],
                    'mask_path': masks[stem],
                }
            )

    return rows


train_pairs = collect_pairs(TRAIN_ROOT)
eval_pairs = collect_pairs(EVAL_ROOT)

print('Train pairs:', len(train_pairs))
print(f'{EVAL_SPLIT_NAME.title()} pairs:', len(eval_pairs))
print('Example train pair:', train_pairs[0] if train_pairs else 'No train pairs found')

Train pairs: 1912
Val pairs: 338
Example train pair: {'case_id': 'glioma_BraTS_GLI_00000_000', 'tumor_type': 'glioma', 'stem': 'BraTS-GLI-00000-000', 'image_path': PosixPath('/content/drive/MyDrive/Brats_Final_Data/Processed_BraTS_128/train/glioma/BraTS-GLI-00000-000_image.nii.gz'), 'mask_path': PosixPath('/content/drive/MyDrive/Brats_Final_Data/Processed_BraTS_128/train/glioma/BraTS-GLI-00000-000_mask.nii.gz')}


## 4. Inspect labels, class balance, and T1c-only source sanity

In [4]:
class_counts = Counter(row['tumor_type'] for row in train_pairs)
print('Training class counts:', dict(class_counts))


def load_mask_consecutive(path: Path) -> np.ndarray:
    mask = np.asanyarray(nib.load(str(path)).dataobj).astype(np.int16)
    if mask.ndim == 4 and mask.shape[-1] == 1:
        mask = mask[..., 0]
    if 4 in np.unique(mask) and 3 not in np.unique(mask):
        mask = mask.copy()
        mask[mask == 4] = 3
    return mask


def discover_mask_values(pairs: list[dict]) -> list[int]:
    values = set()
    for row in pairs:
        values.update(int(v) for v in np.unique(load_mask_consecutive(row['mask_path'])))
    return sorted(values)


mask_values = discover_mask_values(train_pairs)
print('Mask values found:', mask_values)

expected_values = {0, 1, 2, 3}
if not set(mask_values).issubset(expected_values):
    raise ValueError(f'Unexpected mask values: {mask_values}. Expected subset of {sorted(expected_values)}')

label_value_names = {
    0: 'background',
    1: 'edema',
    2: 'non_enhancing_tumor',
    3: 'enhancing_tumor',
}
print('Raw label semantics:', {label_value_names[v]: v for v in mask_values})

region_presence = {
    'whole_tumor': 0,
    'tumor_core': 0,
    'enhancing_tumor': 0,
}

for row in train_pairs:
    mask = load_mask_consecutive(row['mask_path'])
    if np.any(np.isin(mask, [1, 2, 3])):
        region_presence['whole_tumor'] += 1
    if np.any(np.isin(mask, [2, 3])):
        region_presence['tumor_core'] += 1
    if np.any(mask == 3):
        region_presence['enhancing_tumor'] += 1

print('Cases containing each BraTS evaluation region:', region_presence)

if train_pairs:
    sample = train_pairs[0]
    sample_img = nib.load(str(sample['image_path']))
    sample_data = np.asanyarray(sample_img.dataobj)
    sample_mask = load_mask_consecutive(sample['mask_path'])
    print('Sample image shape:', sample_data.shape)
    print('Sample mask shape :', sample_mask.shape)
    if sample_data.ndim != 4:
        raise ValueError(f'Expected a 4-D source image, got shape {sample_data.shape}')
    t1c = sample_data[..., 1]
    print(
        'T1c source index 1 stats:',
        {
            'min': float(t1c.min()),
            'max': float(t1c.max()),
            'mean': float(t1c.mean()),
            'nonzero_voxels': int(np.count_nonzero(t1c)),
        }
    )
    if sample_data.shape[:3] != sample_mask.shape[:3]:
        raise ValueError('Image and mask spatial shapes do not match for the sample case.')

Training class counts: {'glioma': 1062, 'meningioma': 850}
Mask values found: [0, 1, 2, 3]
Raw label semantics: {'background': 0, 'edema': 1, 'non_enhancing_tumor': 2, 'enhancing_tumor': 3}
Cases containing each BraTS evaluation region: {'whole_tumor': 1912, 'tumor_core': 1912, 'enhancing_tumor': 1884}
Sample image shape: (128, 128, 128, 4)
Sample mask shape : (128, 128, 128)
T1c source index 1 stats: {'min': -2.403639078140259, 'max': 12.087651252746582, 'mean': -7.087510311976075e-08, 'nonzero_voxels': 950145}


## 5. Export a direct nnU-Net raw dataset using only T1c

In [5]:
def reset_dir(path: Path) -> None:
    if path.exists():
        shutil.rmtree(path)
    path.mkdir(parents=True, exist_ok=True)


def save_clean_3d_image(src: Path, dst: Path, volume_index: int) -> tuple[tuple[int, ...], tuple[float, float, float]]:
    img = nib.load(str(src))
    data = np.asanyarray(img.dataobj)
    if data.ndim != 4:
        raise ValueError(f'Expected 4-D image for {src.name}, got shape {data.shape}')
    data = np.asarray(data[..., volume_index], dtype=np.float32)
    out = nib.Nifti1Image(data, img.affine)
    out.header.set_zooms(img.header.get_zooms()[:3])
    dst.parent.mkdir(parents=True, exist_ok=True)
    nib.save(out, str(dst))
    return tuple(data.shape), tuple(float(v) for v in out.header.get_zooms()[:3])


def save_clean_mask(src: Path, dst: Path) -> tuple[tuple[int, ...], tuple[float, float, float]]:
    img = nib.load(str(src))
    data = load_mask_consecutive(src)
    if data.ndim != 3:
        raise ValueError(f'Expected 3-D mask for {src.name}, got shape {data.shape}')
    data = np.asarray(data, dtype=np.uint8)
    out = nib.Nifti1Image(data, img.affine)
    out.header.set_zooms(img.header.get_zooms()[:3])
    dst.parent.mkdir(parents=True, exist_ok=True)
    nib.save(out, str(dst))
    return tuple(data.shape), tuple(float(v) for v in out.header.get_zooms()[:3])


def export_split(rows: list[dict], destination_dir: Path, include_labels: bool) -> dict:
    stats = {
        'cases': 0,
        'channel_files': 0,
        'label_files': 0,
        'bad_cases': [],
    }

    for row in rows:
        case_id = row['case_id']
        try:
            image_shapes = []
            image_spacings = []
            for exported_suffix, modality_name, source_index in SOURCE_CHANNELS:
                dst = destination_dir / f'{case_id}_{exported_suffix}.nii.gz'
                shape, spacing = save_clean_3d_image(
                    row['image_path'],
                    dst,
                    volume_index=source_index,
                )
                image_shapes.append(shape)
                image_spacings.append(spacing)
                stats['channel_files'] += 1

            if len(set(image_shapes)) != 1:
                raise ValueError(f'Channel shape mismatch for {case_id}: {image_shapes}')
            if len(set(image_spacings)) != 1:
                raise ValueError(f'Channel spacing mismatch for {case_id}: {image_spacings}')

            if include_labels:
                label_dst = LABELS_TR / f'{case_id}.nii.gz'
                label_shape, label_spacing = save_clean_mask(row['mask_path'], label_dst)
                if label_shape != image_shapes[0]:
                    raise ValueError(
                        f'Label shape mismatch for {case_id}: image={image_shapes[0]}, label={label_shape}'
                    )
                if label_spacing != image_spacings[0]:
                    print(
                        f'Warning: spacing differs for {case_id}: image={image_spacings[0]}, label={label_spacing}'
                    )
                stats['label_files'] += 1

            stats['cases'] += 1
        except Exception as exc:
            stats['bad_cases'].append((case_id, str(exc)))

    return stats


reset_dir(DATASET_ROOT)
for folder in (IMAGES_TR, LABELS_TR, IMAGES_TS):
    folder.mkdir(parents=True, exist_ok=True)

train_export_stats = export_split(train_pairs, IMAGES_TR, include_labels=True)
eval_export_stats = export_split(eval_pairs, IMAGES_TS, include_labels=False)

print('Train export stats:', train_export_stats)
print('Eval export stats :', eval_export_stats)

if train_export_stats['bad_cases'] or eval_export_stats['bad_cases']:
    raise RuntimeError(
        'Export failed for some cases. Inspect train_export_stats and eval_export_stats before continuing.'
    )

Train export stats: {'cases': 1912, 'channel_files': 1912, 'label_files': 1912, 'bad_cases': []}
Eval export stats : {'cases': 338, 'channel_files': 338, 'label_files': 0, 'bad_cases': []}


## 6. Write `dataset.json` for BraTS region-based training and verify the exported dataset

In [6]:
dataset_json = {
    'channel_names': {'0': 'T1c'},
    'labels': {
        'background': 0,
        'whole_tumor': [1, 2, 3],
        'tumor_core': [2, 3],
        'enhancing_tumor': 3,
    },
    'regions_class_order': [1, 2, 3],
    'numTraining': train_export_stats['cases'],
    'file_ending': '.nii.gz',
}

dataset_json_path = DATASET_ROOT / 'dataset.json'
dataset_json_path.write_text(json.dumps(dataset_json, indent=2, sort_keys=False), encoding='utf-8')

train_channel0 = sorted(IMAGES_TR.glob('*_0000.nii.gz'))
train_labels = sorted(LABELS_TR.glob('*.nii.gz'))
eval_channel0 = sorted(IMAGES_TS.glob('*_0000.nii.gz'))

print('dataset.json path:', dataset_json_path)
print(dataset_json_path.read_text(encoding='utf-8'))
print('Training T1c files :', len(train_channel0))
print('Training labels    :', len(train_labels))
print(f'{EVAL_SPLIT_NAME.title()} T1c files:', len(eval_channel0))

if not (len(train_channel0) == len(train_labels) == train_export_stats['cases']):
    raise ValueError('Training export counts do not match.')

sample_case_id = train_channel0[0].name.replace('_0000.nii.gz', '')
sample_t1c = np.asanyarray(nib.load(str(IMAGES_TR / f'{sample_case_id}_0000.nii.gz')).dataobj)
sample_label = np.asanyarray(nib.load(str(LABELS_TR / f'{sample_case_id}.nii.gz')).dataobj)
print('Verified sample case:', sample_case_id)
print('T1c shape   :', sample_t1c.shape)
print('Label shape :', sample_label.shape)

if sample_t1c.ndim != 3 or sample_label.ndim != 3:
    raise ValueError('Exported files must be 3-D NIfTI volumes.')
if sample_t1c.shape != sample_label.shape:
    raise ValueError('Exported image and label shapes do not match.')

dataset.json path: /content/brats_workspace/nnUNet_raw/Dataset303_ProcessedBraTS128_T1c_Regions/dataset.json
{
  "channel_names": {
    "0": "T1c"
  },
  "labels": {
    "background": 0,
    "whole_tumor": [
      1,
      2,
      3
    ],
    "tumor_core": [
      2,
      3
    ],
    "enhancing_tumor": 3
  },
  "regions_class_order": [
    1,
    2,
    3
  ],
  "numTraining": 1912,
  "file_ending": ".nii.gz"
}
Training T1c files : 1912
Training labels    : 1912
Val T1c files: 338
Verified sample case: glioma_BraTS_GLI_00000_000
T1c shape   : (128, 128, 128)
Label shape : (128, 128, 128)


## 7. Set nnU-Net environment variables

In [7]:
os.environ['nnUNet_raw'] = str(NNUNET_RAW)
os.environ['nnUNet_preprocessed'] = str(NNUNET_PREPROCESSED)
os.environ['nnUNet_results'] = str(NNUNET_RESULTS)
os.environ['BRATS_CHECKPOINT_MIRROR'] = str(CHECKPOINT_MIRROR_DIR)


def install_budget_trainer() -> Path:
    import nnunetv2

    trainer_dir = Path(nnunetv2.__file__).parent / 'training' / 'nnUNetTrainer'
    trainer_path = trainer_dir / f'{TRAINER_NAME}.py'

    trainer_code = f'''
import json
import os
import shutil
from pathlib import Path

import torch

from nnunetv2.training.nnUNetTrainer.nnUNetTrainer import nnUNetTrainer


class {TRAINER_NAME}(nnUNetTrainer):
    def __init__(self, plans, configuration, fold, dataset_json, device=torch.device('cuda')):
        super().__init__(plans, configuration, fold, dataset_json, device)
        self.num_epochs = {MAX_EPOCHS}
        self.save_every = {SAVE_EVERY_N_EPOCHS}

    def save_checkpoint(self, filename: str, *args, **kwargs) -> None:
        super().save_checkpoint(filename, *args, **kwargs)

        mirror_root = os.environ.get('BRATS_CHECKPOINT_MIRROR')
        if not mirror_root:
            return

        source = Path(filename)
        if not source.is_absolute():
            source = Path(self.output_folder) / filename
        if not source.exists():
            return

        target_dir = Path(mirror_root) / f'fold_{{self.fold}}'
        target_dir.mkdir(parents=True, exist_ok=True)

        target = target_dir / source.name
        shutil.copy2(source, target)

        state = {{
            'fold': int(self.fold),
            'epoch': int(getattr(self, 'current_epoch', -1)),
            'checkpoint': source.name,
            'mirrored_to': str(target),
            'output_folder': str(self.output_folder),
        }}
        (target_dir / 'latest_mirror_state.json').write_text(
            json.dumps(state, indent=2),
            encoding='utf-8'
        )
'''.lstrip()

    trainer_path.write_text(trainer_code, encoding='utf-8')
    return trainer_path


budget_trainer_path = install_budget_trainer()
CHECKPOINT_MIRROR_DIR.mkdir(parents=True, exist_ok=True)

print('Budget trainer    =', budget_trainer_path)
print('Checkpoint mirror =', CHECKPOINT_MIRROR_DIR)
print('Trainer fixed     = exact base signature: plans, configuration, fold, dataset_json, device')


Budget trainer    = /usr/local/lib/python3.12/dist-packages/nnunetv2/training/nnUNetTrainer/nnUNetTrainerBudget250Epochs.py
Checkpoint mirror = /content/drive/MyDrive/gp2/checkpoints/Dataset303_ProcessedBraTS128_T1c_Regions/nnUNetTrainerBudget250Epochs__nnUNetResEncUNetMPlans__3d_fullres
Trainer fixed     = exact base signature: plans, configuration, fold, dataset_json, device


## 8. Plan and preprocess with the stronger ResEnc planner

In [8]:
command = [
    'nnUNetv2_plan_and_preprocess',
    '-d', str(DATASET_ID),
    '--verify_dataset_integrity',
    '-pl', PLANNER_NAME,
]

print('Running:', ' '.join(command))
result = subprocess.run(command, text=True, capture_output=True, env=os.environ.copy())

print(result.stdout)
if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError(f'nnUNetv2_plan_and_preprocess failed with exit code {result.returncode}')

print('Preprocessing finished.')
print('Preprocessed root:', NNUNET_PREPROCESSED / DATASET_FOLDER)

Running: nnUNetv2_plan_and_preprocess -d 303 --verify_dataset_integrity -pl nnUNetPlannerResEncM
Fingerprint extraction...
Dataset303_ProcessedBraTS128_T1c_Regions
Using <class 'nnunetv2.imageio.simpleitk_reader_writer.SimpleITKIO'> as reader/writer

####################
verify_dataset_integrity Done. 
If you didn't see any error messages then your dataset is most likely OK!
####################

Using <class 'nnunetv2.imageio.simpleitk_reader_writer.SimpleITKIO'> as reader/writer
Experiment planning...
Dropping 3d_lowres config because the image size difference to 3d_fullres is too small. 3d_fullres: [128. 128. 128.], 3d_lowres: [128, 128, 128]
2D U-Net configuration:
{'data_identifier': 'nnUNetPlans_2d', 'preprocessor_name': 'DefaultPreprocessor', 'batch_size': 201, 'patch_size': (np.int64(128), np.int64(128)), 'median_image_size_in_voxels': array([128., 128.]), 'spacing': array([1.0859375, 1.0546875]), 'normalization_schemes': ['ZScoreNormalization'], 'use_mask_for_norm': [False], '

## 9. Train the budget fold with every-epoch checkpoints

In [ ]:
def run_streaming_command(command: list[str]) -> None:
    print('Running:', ' '.join(command))
    popen_kwargs = {
        'stdout': subprocess.PIPE,
        'stderr': subprocess.STDOUT,
        'text': True,
        'bufsize': 1,
        'env': os.environ.copy(),
    }
    if os.name != 'nt':
        popen_kwargs['preexec_fn'] = os.setsid

    proc = subprocess.Popen(command, **popen_kwargs)
    try:
        assert proc.stdout is not None
        for line in proc.stdout:
            print(line, end='')
        proc.wait()
    except KeyboardInterrupt:
        if os.name == 'nt':
            proc.terminate()
        else:
            os.killpg(os.getpgid(proc.pid), signal.SIGTERM)
        print('\nCommand interrupted cleanly. The previous completed epoch should be available as checkpoint_latest.pth.')
        raise
    if proc.returncode not in (0, -15):
        raise RuntimeError(f'Command failed with exit code {proc.returncode}')


def fold_dir(fold: int) -> Path:
    return MODEL_BASE / f'fold_{fold}'


def fold_mirror_dir(fold: int) -> Path:
    return CHECKPOINT_MIRROR_DIR / f'fold_{fold}'


def fold_is_finished(fold: int) -> bool:
    return (fold_dir(fold) / 'checkpoint_final.pth').exists()


def fold_has_resume_checkpoint(fold: int) -> bool:
    return (fold_dir(fold) / 'checkpoint_latest.pth').exists()


print(f'Budget run: {len(TRAIN_FOLDS)} fold(s), {MAX_EPOCHS} epochs max.')
print(f'Estimated cost: {ESTIMATED_TOTAL_HOURS:.1f} hours / {ESTIMATED_TOTAL_CU:.1f} CU at {A100_COMPUTE_UNITS_PER_HOUR:.2f} CU/hour.')
print('Checkpoints mirror to:', CHECKPOINT_MIRROR_DIR)

for fold in TRAIN_FOLDS:
    out_dir = fold_dir(fold)
    out_dir.mkdir(parents=True, exist_ok=True)
    fold_mirror_dir(fold).mkdir(parents=True, exist_ok=True)

    if fold_is_finished(fold):
        print(f'Fold {fold} already finished, skipping.')
        continue

    train_command = [
        'nnUNetv2_train',
        str(DATASET_ID),
        CONFIGURATION,
        str(fold),
        '-tr', TRAINER_NAME,
        '-p', PLANS_NAME,
        '--npz',
    ]

    if fold_has_resume_checkpoint(fold):
        train_command.append('--c')
        print(f'Resuming fold {fold} from checkpoint_latest.pth')
    else:
        print(f'Starting fold {fold} from scratch')

    run_streaming_command(train_command)

print('Training stage completed.')
for fold in TRAIN_FOLDS:
    checkpoints = [p.name for p in sorted(fold_dir(fold).glob('checkpoint_*'))]
    mirrored = [p.name for p in sorted(fold_mirror_dir(fold).glob('checkpoint_*'))]
    print(f'Fold {fold} result checkpoints:', checkpoints)
    print(f'Fold {fold} mirrored checkpoints:', mirrored)

Budget run: 1 fold(s), 250 epochs max.
Estimated cost: 80.0 hours / 429.6 CU at 5.37 CU/hour.
Checkpoints mirror to: /content/drive/MyDrive/gp2/checkpoints/Dataset303_ProcessedBraTS128_T1c_Regions/nnUNetTrainerBudget100Epochs__nnUNetResEncUNetMPlans__3d_fullres
Starting fold 0 from scratch
Running: nnUNetv2_train 303 3d_fullres 0 -tr nnUNetTrainerBudget250Epochs -p nnUNetResEncUNetMPlans --npz
Using device: cuda:0

#######################################################################
Please cite the following paper when using nnU-Net:
Isensee, F., Jaeger, P. F., Kohl, S. A., Petersen, J., & Maier-Hein, K. H. (2021). nnU-Net: a self-configuring method for deep learning-based biomedical image segmentation. Nature methods, 18(2), 203-211.
#######################################################################

2026-04-24 20:15:48.096806: Using torch.compile...
2026-04-24 20:15:49.442537: do_dummy_2d_data_aug: False
2026-04-24 20:15:49.448322: Using splits from existing split file: /cont

In [9]:
from pathlib import Path

ckpt_dir = Path('/content/drive/MyDrive/gp2/checkpoints/Dataset303_ProcessedBraTS128_T1c_Regions/nnUNetTrainerBudget100Epochs__nnUNetResEncUNetMPlans__3d_fullres/fold_0')

print(ckpt_dir)
print('Exists:', ckpt_dir.exists())
print([p.name for p in ckpt_dir.glob('checkpoint_*.pth')])


/content/drive/MyDrive/gp2/checkpoints/Dataset303_ProcessedBraTS128_T1c_Regions/nnUNetTrainerBudget100Epochs__nnUNetResEncUNetMPlans__3d_fullres/fold_0
Exists: True
['checkpoint_best.pth', 'checkpoint_latest.pth', 'checkpoint_final.pth']


## 10. Run out-of-fold validation prediction and compute official BraTS regional Dice/HD95

In [10]:
from scipy.spatial import cKDTree


def choose_checkpoint_name(fold: int) -> str:
    candidates = [
        'checkpoint_best.pth',
        'checkpoint_final.pth',
        'checkpoint_latest.pth',
    ]
    current_fold_dir = fold_dir(fold)
    for name in candidates:
        if (current_fold_dir / name).exists():
            return name
    raise FileNotFoundError(f'No checkpoint found in {current_fold_dir}')


def region_masks(seg: np.ndarray) -> dict[str, np.ndarray]:
    return {
        'whole_tumor': np.isin(seg, [1, 2, 3]),
        'tumor_core': np.isin(seg, [2, 3]),
        'enhancing_tumor': seg == 3,
    }


def dice_score(pred: np.ndarray, gt: np.ndarray) -> float:
    inter = int((pred & gt).sum())
    total = int(pred.sum() + gt.sum())
    return float(2 * inter / total) if total > 0 else float('nan')


def hd95(pred: np.ndarray, gt: np.ndarray, voxel_spacing: tuple[float, float, float]) -> float:
    pred_pts = np.argwhere(pred) * voxel_spacing
    gt_pts = np.argwhere(gt) * voxel_spacing
    if len(pred_pts) == 0 or len(gt_pts) == 0:
        return float('nan')
    tree_gt = cKDTree(gt_pts)
    tree_pred = cKDTree(pred_pts)
    d_p2g = tree_gt.query(pred_pts)[0]
    d_g2p = tree_pred.query(gt_pts)[0]
    return float(np.percentile(np.concatenate([d_p2g, d_g2p]), 95))


splits_path = NNUNET_PREPROCESSED / DATASET_FOLDER / 'splits_final.json'
splits = json.loads(splits_path.read_text(encoding='utf-8'))

OOF_INPUT_ROOT = WORKSPACE_DIR / 'oof_inputs'
OOF_FOLD_ROOT = WORKSPACE_DIR / 'oof_predictions_per_fold'
OOF_MERGED_DIR = WORKSPACE_DIR / 'oof_predictions_merged'

reset_dir(OOF_INPUT_ROOT)
reset_dir(OOF_FOLD_ROOT)
reset_dir(OOF_MERGED_DIR)

for fold in TRAIN_FOLDS:
    val_ids = sorted(splits[fold]['val'])
    fold_input_dir = OOF_INPUT_ROOT / f'fold_{fold}'
    fold_output_dir = OOF_FOLD_ROOT / f'fold_{fold}'
    reset_dir(fold_input_dir)
    reset_dir(fold_output_dir)

    linked = 0
    for val_id in val_ids:
        for channel_file in IMAGES_TR.glob(f'{val_id}_*.nii.gz'):
            dst = fold_input_dir / channel_file.name
            try:
                dst.symlink_to(channel_file)
            except OSError:
                shutil.copy2(channel_file, dst)
            linked += 1

    checkpoint_name = choose_checkpoint_name(fold)
    predict_command = [
        'nnUNetv2_predict',
        '-i', str(fold_input_dir),
        '-o', str(fold_output_dir),
        '-d', str(DATASET_ID),
        '-c', CONFIGURATION,
        '-tr', TRAINER_NAME,
        '-p', PLANS_NAME,
        '-f', str(fold),
        '-chk', checkpoint_name,
    ]

    print(f'Fold {fold}: {len(val_ids)} validation cases, {linked} input channel files')
    result = subprocess.run(
        predict_command,
        text=True,
        capture_output=True,
        env=os.environ.copy(),
    )
    print(result.stdout[-2000:] if len(result.stdout) > 2000 else result.stdout)
    if result.returncode != 0:
        print(result.stderr[-2000:] if len(result.stderr) > 2000 else result.stderr)
        raise RuntimeError(f'Validation prediction failed for fold {fold} with exit code {result.returncode}')

    for pred_file in fold_output_dir.glob('*.nii.gz'):
        shutil.copy2(pred_file, OOF_MERGED_DIR / pred_file.name)

results = {
    'whole_tumor': {'dice': [], 'hd95': []},
    'tumor_core': {'dice': [], 'hd95': []},
    'enhancing_tumor': {'dice': [], 'hd95': []},
}

pred_files = sorted(OOF_MERGED_DIR.glob('*.nii.gz'))
print(f'Evaluating {len(pred_files)} out-of-fold predictions...')

for i, pred_file in enumerate(pred_files, start=1):
    gt_file = LABELS_TR / pred_file.name
    if not gt_file.exists():
        print('Missing ground truth for', pred_file.name)
        continue

    pred_img = nib.load(str(pred_file))
    gt_img = nib.load(str(gt_file))
    pred_vol = np.asanyarray(pred_img.dataobj).astype(np.int16)
    gt_vol = np.asanyarray(gt_img.dataobj).astype(np.int16)
    spacing = tuple(float(v) for v in pred_img.header.get_zooms()[:3])

    pred_regions = region_masks(pred_vol)
    gt_regions = region_masks(gt_vol)

    for region_name in results:
        results[region_name]['dice'].append(dice_score(pred_regions[region_name], gt_regions[region_name]))
        results[region_name]['hd95'].append(hd95(pred_regions[region_name], gt_regions[region_name], spacing))

    if i % 100 == 0:
        print(f'  {i}/{len(pred_files)} cases evaluated...')

print(f"\n{'Region':<18} {'Mean Dice':>10} {'Mean HD95 (mm)':>15}")
print('-' * 48)
overall_dice = []
overall_hd95 = []
for region_name in ['whole_tumor', 'tumor_core', 'enhancing_tumor']:
    mean_dice = float(np.nanmean(results[region_name]['dice']))
    mean_hd95 = float(np.nanmean(results[region_name]['hd95']))
    overall_dice.append(mean_dice)
    overall_hd95.append(mean_hd95)
    print(f'{region_name:<18} {mean_dice:>10.4f} {mean_hd95:>15.2f}')
print('-' * 48)
print(f"{'Mean':<18} {float(np.nanmean(overall_dice)):>10.4f} {float(np.nanmean(overall_hd95)):>15.2f}")

FileNotFoundError: [Errno 2] No such file or directory: '/content/brats_workspace/nnUNet_preprocessed/Dataset303_ProcessedBraTS128_T1c_Regions/splits_final.json'

In [16]:
import json, os, shutil, subprocess
from pathlib import Path

import nibabel as nib
import numpy as np
from scipy.spatial import cKDTree
from sklearn.model_selection import KFold

TRAINER_NAME = 'nnUNetTrainerBudget100Epochs'
PLANS_NAME = 'nnUNetResEncUNetMPlans'
CONFIGURATION = '3d_fullres'

DRIVE_CKPT_ROOT = Path('/content/drive/MyDrive/gp2/checkpoints')

os.environ['nnUNet_results'] = str(DRIVE_CKPT_ROOT)

MODEL_BASE = (
    DRIVE_CKPT_ROOT
    / DATASET_FOLDER
    / f'{TRAINER_NAME}__{PLANS_NAME}__{CONFIGURATION}'
)

def fold_dir(fold: int) -> Path:
    return MODEL_BASE / f'fold_{fold}'

print('Using checkpoint root:', MODEL_BASE)
print('Fold 0 checkpoints:', sorted(p.name for p in fold_dir(0).glob('checkpoint_*.pth')))

MODEL_BASE.mkdir(parents=True, exist_ok=True)

dataset_json_src = DATASET_ROOT / 'dataset.json'
plans_json_src = NNUNET_PREPROCESSED / DATASET_FOLDER / f'{PLANS_NAME}.json'

dataset_json_dst = MODEL_BASE / 'dataset.json'
plans_json_dst = MODEL_BASE / 'plans.json'

if not dataset_json_src.exists():
    raise FileNotFoundError(f'Missing source dataset.json: {dataset_json_src}')

if not plans_json_src.exists():
    raise FileNotFoundError(f'Missing source plans file: {plans_json_src}')

shutil.copy2(dataset_json_src, dataset_json_dst)
shutil.copy2(plans_json_src, plans_json_dst)

print('Copied model metadata:')
print(' ', dataset_json_dst)
print(' ', plans_json_dst)
print('Model folder now has:', sorted(p.name for p in MODEL_BASE.iterdir()))

def choose_checkpoint_name(fold: int) -> str:
    candidates = [
        'checkpoint_latest.pth',
        'checkpoint_final.pth',
        'checkpoint_best.pth',
    ]
    current_fold_dir = fold_dir(fold)
    for name in candidates:
        if (current_fold_dir / name).exists():
            return name
    raise FileNotFoundError(f'No checkpoint found in {current_fold_dir}')



def region_masks(seg: np.ndarray) -> dict[str, np.ndarray]:
    return {
        'whole_tumor': np.isin(seg, [1, 2, 3]),
        'tumor_core': np.isin(seg, [2, 3]),
        'enhancing_tumor': seg == 3,
    }


def dice_score(pred: np.ndarray, gt: np.ndarray) -> float:
    inter = int((pred & gt).sum())
    total = int(pred.sum() + gt.sum())
    return float(2 * inter / total) if total > 0 else float('nan')


def hd95(pred: np.ndarray, gt: np.ndarray, voxel_spacing: tuple[float, float, float]) -> float:
    pred_pts = np.argwhere(pred) * voxel_spacing
    gt_pts = np.argwhere(gt) * voxel_spacing
    if len(pred_pts) == 0 or len(gt_pts) == 0:
        return float('nan')
    tree_gt = cKDTree(gt_pts)
    tree_pred = cKDTree(pred_pts)
    d_p2g = tree_gt.query(pred_pts)[0]
    d_g2p = tree_pred.query(gt_pts)[0]
    return float(np.percentile(np.concatenate([d_p2g, d_g2p]), 95))


def load_or_recreate_splits() -> list[dict[str, list[str]]]:
    splits_path = NNUNET_PREPROCESSED / DATASET_FOLDER / 'splits_final.json'

    if splits_path.exists():
        print('Using existing splits:', splits_path)
        return json.loads(splits_path.read_text(encoding='utf-8'))

    print('splits_final.json missing, recreating nnU-Net default 5-fold split...')
    print('Missing path was:', splits_path)

    case_ids = sorted(p.name[:-7] for p in LABELS_TR.glob('*.nii.gz'))
    if not case_ids:
        raise FileNotFoundError(f'No label files found in {LABELS_TR}')

    kfold = KFold(n_splits=5, shuffle=True, random_state=12345)
    splits = []

    for train_idx, val_idx in kfold.split(case_ids):
        splits.append({
            'train': [case_ids[i] for i in train_idx],
            'val': [case_ids[i] for i in val_idx],
        })

    splits_path.parent.mkdir(parents=True, exist_ok=True)
    splits_path.write_text(json.dumps(splits, indent=2), encoding='utf-8')
    print('Recreated splits at:', splits_path)

    return splits


splits = load_or_recreate_splits()

OOF_INPUT_ROOT = WORKSPACE_DIR / 'oof_inputs'
OOF_FOLD_ROOT = WORKSPACE_DIR / 'oof_predictions_per_fold'
OOF_MERGED_DIR = WORKSPACE_DIR / 'oof_predictions_merged'

reset_dir(OOF_INPUT_ROOT)
reset_dir(OOF_FOLD_ROOT)
reset_dir(OOF_MERGED_DIR)

for fold in TRAIN_FOLDS:
    val_ids = sorted(splits[fold]['val'])
    fold_input_dir = OOF_INPUT_ROOT / f'fold_{fold}'
    fold_output_dir = OOF_FOLD_ROOT / f'fold_{fold}'
    reset_dir(fold_input_dir)
    reset_dir(fold_output_dir)

    linked = 0
    missing_inputs = []

    for val_id in val_ids:
        channel_files = sorted(IMAGES_TR.glob(f'{val_id}_*.nii.gz'))
        if not channel_files:
            missing_inputs.append(val_id)
            continue

        for channel_file in channel_files:
            dst = fold_input_dir / channel_file.name
            try:
                dst.symlink_to(channel_file)
            except OSError:
                shutil.copy2(channel_file, dst)
            linked += 1

    if missing_inputs:
        print(f'Warning: {len(missing_inputs)} validation cases had no input files.')
        print('First missing cases:', missing_inputs[:10])

    checkpoint_name = choose_checkpoint_name(fold)

    predict_command = [
        'nnUNetv2_predict',
        '-i', str(fold_input_dir),
        '-o', str(fold_output_dir),
        '-d', str(DATASET_ID),
        '-c', CONFIGURATION,
        '-tr', TRAINER_NAME,
        '-p', PLANS_NAME,
        '-f', str(fold),
        '-chk', checkpoint_name,
    ]

    print(f'\nFold {fold}: {len(val_ids)} validation cases, {linked} input channel files')
    print('Checkpoint:', checkpoint_name)
    print('Running:', ' '.join(predict_command))

    result = subprocess.run(
        predict_command,
        text=True,
        capture_output=True,
        env=os.environ.copy(),
    )

    print(result.stdout[-2000:] if len(result.stdout) > 2000 else result.stdout)

    if result.returncode != 0:
        print(result.stderr[-4000:] if len(result.stderr) > 4000 else result.stderr)
        raise RuntimeError(f'Validation prediction failed for fold {fold} with exit code {result.returncode}')

    for pred_file in fold_output_dir.glob('*.nii.gz'):
        shutil.copy2(pred_file, OOF_MERGED_DIR / pred_file.name)


results = {
    'whole_tumor': {'dice': [], 'hd95': []},
    'tumor_core': {'dice': [], 'hd95': []},
    'enhancing_tumor': {'dice': [], 'hd95': []},
}

pred_files = sorted(OOF_MERGED_DIR.glob('*.nii.gz'))
print(f'\nEvaluating {len(pred_files)} out-of-fold predictions...')

for i, pred_file in enumerate(pred_files, start=1):
    gt_file = LABELS_TR / pred_file.name
    if not gt_file.exists():
        print('Missing ground truth for', pred_file.name)
        continue

    pred_img = nib.load(str(pred_file))
    gt_img = nib.load(str(gt_file))

    pred_vol = np.asanyarray(pred_img.dataobj).astype(np.int16)
    gt_vol = np.asanyarray(gt_img.dataobj).astype(np.int16)
    spacing = tuple(float(v) for v in pred_img.header.get_zooms()[:3])

    pred_regions = region_masks(pred_vol)
    gt_regions = region_masks(gt_vol)

    for region_name in results:
        results[region_name]['dice'].append(
            dice_score(pred_regions[region_name], gt_regions[region_name])
        )
        results[region_name]['hd95'].append(
            hd95(pred_regions[region_name], gt_regions[region_name], spacing)
        )

    if i % 100 == 0:
        print(f'  {i}/{len(pred_files)} cases evaluated...')


print(f"\n{'Region':<18} {'Mean Dice':>10} {'Mean HD95 (mm)':>15}")
print('-' * 48)

overall_dice = []
overall_hd95 = []

for region_name in ['whole_tumor', 'tumor_core', 'enhancing_tumor']:
    mean_dice = float(np.nanmean(results[region_name]['dice']))
    mean_hd95 = float(np.nanmean(results[region_name]['hd95']))

    overall_dice.append(mean_dice)
    overall_hd95.append(mean_hd95)

    print(f'{region_name:<18} {mean_dice:>10.4f} {mean_hd95:>15.2f}')

print('-' * 48)
print(f"{'Mean':<18} {float(np.nanmean(overall_dice)):>10.4f} {float(np.nanmean(overall_hd95)):>15.2f}")



Using checkpoint root: /content/drive/MyDrive/gp2/checkpoints/Dataset303_ProcessedBraTS128_T1c_Regions/nnUNetTrainerBudget100Epochs__nnUNetResEncUNetMPlans__3d_fullres
Fold 0 checkpoints: ['checkpoint_best.pth', 'checkpoint_final.pth', 'checkpoint_latest.pth']
Copied model metadata:
  /content/drive/MyDrive/gp2/checkpoints/Dataset303_ProcessedBraTS128_T1c_Regions/nnUNetTrainerBudget100Epochs__nnUNetResEncUNetMPlans__3d_fullres/dataset.json
  /content/drive/MyDrive/gp2/checkpoints/Dataset303_ProcessedBraTS128_T1c_Regions/nnUNetTrainerBudget100Epochs__nnUNetResEncUNetMPlans__3d_fullres/plans.json
Model folder now has: ['dataset.json', 'fold_0', 'plans.json']
Using existing splits: /content/brats_workspace/nnUNet_preprocessed/Dataset303_ProcessedBraTS128_T1c_Regions/splits_final.json

Fold 0: 383 validation cases, 383 input channel files
Checkpoint: checkpoint_latest.pth
Running: nnUNetv2_predict -i /content/brats_workspace/oof_inputs/fold_0 -o /content/brats_workspace/oof_predictions_per

## 11. Predict on the evaluation split with all available folds

In [18]:
def choose_common_checkpoint_name(folds: list[int]) -> str:
    candidates = [
        'checkpoint_best.pth',
        'checkpoint_final.pth',
        'checkpoint_latest.pth',
    ]
    for name in candidates:
        if all((fold_dir(fold) / name).exists() for fold in folds):
            return name
    raise FileNotFoundError(f'No common checkpoint found across folds {folds}')


available_folds = [fold for fold in TRAIN_FOLDS if fold_dir(fold).exists()]
if not available_folds:
    raise FileNotFoundError('No trained folds found. Run the training cell first.')

checkpoint_name = choose_common_checkpoint_name(available_folds)
PREDICTION_DIR = WORKSPACE_DIR / f'predictions_{EVAL_SPLIT_NAME}'
reset_dir(PREDICTION_DIR)

predict_command = [
    'nnUNetv2_predict',
    '-i', str(IMAGES_TS),
    '-o', str(PREDICTION_DIR),
    '-d', str(DATASET_ID),
    '-c', CONFIGURATION,
    '-tr', TRAINER_NAME,
    '-p', PLANS_NAME,
    '-f', *[str(fold) for fold in available_folds],
    '-chk', checkpoint_name,
]

print('Running:', ' '.join(predict_command))
print('Prediction folds:', available_folds)
print('Prediction checkpoint:', checkpoint_name)
result = subprocess.run(
    predict_command,
    text=True,
    capture_output=True,
    env=os.environ.copy(),
)

print(result.stdout)
if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError(f'Prediction failed with exit code {result.returncode}')

print('Prediction finished.')
print('Folds used      :', available_folds)
print('Checkpoint used :', checkpoint_name)
print('Prediction output:', PREDICTION_DIR)

FileNotFoundError: No trained folds found. Run the training cell first.

## 13. Visualize a sample prediction on T1c

In [ ]:
import matplotlib.pyplot as plt
from ipywidgets import IntSlider, FloatSlider, Checkbox, VBox, HBox, interactive_output
from IPython.display import display


def load_vol(path: Path) -> np.ndarray:
    vol = np.asanyarray(nib.load(str(path)).dataobj)
    if vol.ndim == 4 and vol.shape[-1] == 1:
        vol = vol[..., 0]
    return vol


def best_indices(mask: np.ndarray | None, fallback_shape: tuple[int, int, int]) -> tuple[int, int, int]:
    if mask is None or not np.any(mask > 0):
        return (
            fallback_shape[0] // 2,
            fallback_shape[1] // 2,
            fallback_shape[2] // 2,
        )
    sagittal = int(np.argmax((mask > 0).sum(axis=(1, 2))))
    coronal = int(np.argmax((mask > 0).sum(axis=(0, 2))))
    axial = int(np.argmax((mask > 0).sum(axis=(0, 1))))
    return sagittal, coronal, axial


def norm_slice(arr: np.ndarray) -> np.ndarray:
    arr = arr.astype(np.float32)
    lo, hi = np.percentile(arr, [1, 99])
    if hi <= lo:
        return arr
    arr = np.clip(arr, lo, hi)
    return (arr - lo) / (hi - lo + 1e-8)


def plane_slice(vol: np.ndarray, plane: str, idx: int) -> np.ndarray:
    if plane == 'axial':
        return np.rot90(vol[:, :, idx])
    if plane == 'coronal':
        return np.rot90(vol[:, idx, :])
    if plane == 'sagittal':
        return np.rot90(vol[idx, :, :])
    raise ValueError(f'Unknown plane: {plane}')


pred_files = sorted(PREDICTION_DIR.glob('*.nii.gz'))
if not pred_files:
    raise FileNotFoundError(f'No predictions found in {PREDICTION_DIR}')

sample_index = 0
sample_pred = pred_files[sample_index]
case_id = sample_pred.name.replace('.nii.gz', '')
sample_img = IMAGES_TS / f'{case_id}_0000.nii.gz'
sample_gt_path = eval_mask_by_case.get(case_id)

print('Case      :', case_id)
print('Image     :', sample_img)
print('Prediction:', sample_pred)
print('Ground tr.:', sample_gt_path if sample_gt_path is not None else 'Not available')

t1c = load_vol(sample_img).astype(np.float32)
pred_mask = load_vol(sample_pred).astype(np.int16)
gt_mask = load_mask_consecutive(sample_gt_path) if sample_gt_path is not None else None

sag0, cor0, ax0 = best_indices(gt_mask if gt_mask is not None else pred_mask, t1c.shape)

sagittal_slider = IntSlider(description='Sagittal', min=0, max=t1c.shape[0] - 1, value=sag0, continuous_update=False)
coronal_slider = IntSlider(description='Coronal', min=0, max=t1c.shape[1] - 1, value=cor0, continuous_update=False)
axial_slider = IntSlider(description='Axial', min=0, max=t1c.shape[2] - 1, value=ax0, continuous_update=False)
alpha_slider = FloatSlider(description='Mask alpha', min=0.0, max=1.0, step=0.05, value=0.45, continuous_update=False)
show_gt_checkbox = Checkbox(description='Show GT row', value=(gt_mask is not None), disabled=(gt_mask is None))


def render_view(sagittal_idx: int, coronal_idx: int, axial_idx: int, alpha: float, show_gt: bool):
    planes = [
        ('sagittal', sagittal_idx),
        ('coronal', coronal_idx),
        ('axial', axial_idx),
    ]

    rows = 2 if show_gt and gt_mask is not None else 1
    fig, axes = plt.subplots(rows, 3, figsize=(18, 5 * rows))
    if rows == 1:
        axes = np.array([axes])

    for col, (plane, idx) in enumerate(planes):
        img_slice = norm_slice(plane_slice(t1c, plane, idx))
        pred_slice = plane_slice(pred_mask, plane, idx)

        axes[0, col].imshow(img_slice, cmap='gray')
        axes[0, col].imshow(pred_slice, cmap='autumn', alpha=(pred_slice > 0) * alpha)
        axes[0, col].set_title(f'{plane.title()} | pred | idx={idx}')
        axes[0, col].axis('off')

        if rows == 2:
            gt_slice = plane_slice(gt_mask, plane, idx)
            axes[1, col].imshow(img_slice, cmap='gray')
            axes[1, col].imshow(gt_slice, cmap='viridis', alpha=(gt_slice > 0) * alpha)
            axes[1, col].set_title(f'{plane.title()} | ground truth | idx={idx}')
            axes[1, col].axis('off')

    plt.tight_layout()
    plt.show()


controls = {
    'sagittal_idx': sagittal_slider,
    'coronal_idx': coronal_slider,
    'axial_idx': axial_slider,
    'alpha': alpha_slider,
    'show_gt': show_gt_checkbox,
}

ui = VBox([
    HBox([sagittal_slider, coronal_slider, axial_slider]),
    HBox([alpha_slider, show_gt_checkbox]),
])
out = interactive_output(render_view, controls)
display(ui, out)


Case      : glioma_BraTS_GLI_00006_000
Image     : /content/brats_workspace/nnUNet_raw/Dataset303_ProcessedBraTS128_T1c_Regions/imagesTs/glioma_BraTS_GLI_00006_000_0000.nii.gz
Prediction: /content/brats_workspace/predictions_val/glioma_BraTS_GLI_00006_000.nii.gz
Ground tr.: /content/drive/MyDrive/Brats_Final_Data/Processed_BraTS_128/val/glioma/BraTS-GLI-00006-000_mask.nii.gz


Output()

## 14. Backup the full workspace to Drive so the model is ready after a runtime restart

In [19]:
import datetime

SRC_WORKSPACE = WORKSPACE_DIR
BACKUP_DIR = BACKUP_ROOT / DATASET_FOLDER
BACKUP_DIR.mkdir(parents=True, exist_ok=True)

items_to_copy = {
    'nnUNet_raw': SRC_WORKSPACE / 'nnUNet_raw',
    'nnUNet_preprocessed': SRC_WORKSPACE / 'nnUNet_preprocessed',
    'nnUNet_results': SRC_WORKSPACE / 'nnUNet_results',
    'oof_predictions_merged': OOF_MERGED_DIR,
    f'predictions_{EVAL_SPLIT_NAME}': PREDICTION_DIR,
}

for name, src in items_to_copy.items():
    if src.exists():
        dst = BACKUP_DIR / name
        if dst.exists():
            shutil.rmtree(dst)
        print(f'Copying {src} -> {dst}')
        shutil.copytree(src, dst)
    else:
        print(f'Skipping missing path: {src}')

restore_info = BACKUP_DIR / 'restore_info.txt'
restore_info.write_text(
    f'''Backup time: {datetime.datetime.now():%Y-%m-%d %H:%M:%S}
DATASET_ID={DATASET_ID}
DATASET_NAME={DATASET_NAME}
DATASET_FOLDER={DATASET_FOLDER}
PLANNER_NAME={PLANNER_NAME}
PLANS_NAME={PLANS_NAME}
TRAINER_NAME={TRAINER_NAME}
CONFIGURATION={CONFIGURATION}
TRAIN_FOLDS={TRAIN_FOLDS}
WORKSPACE_DIR={WORKSPACE_DIR}
''',
    encoding='utf-8'
)

print(f'\nBackup complete: {BACKUP_DIR}')

NameError: name 'PREDICTION_DIR' is not defined

In [21]:
import datetime
import shutil
from pathlib import Path

SRC_WORKSPACE = WORKSPACE_DIR

DRIVE_CKPT_ROOT = Path('/content/drive/MyDrive/gp2/checkpoints')
TRAINED_MODEL_ROOT = DRIVE_CKPT_ROOT / DATASET_FOLDER

BACKUP_ROOT = Path('/content/drive/MyDrive/gp2_full_model_backups')
RUN_STAMP = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
BACKUP_DIR = BACKUP_ROOT / f'{DATASET_FOLDER}_complete_backup_{RUN_STAMP}'
BACKUP_DIR.mkdir(parents=True, exist_ok=False)

items_to_copy = {
    'brats_workspace_complete': SRC_WORKSPACE,
    'trained_model_checkpoints_complete': TRAINED_MODEL_ROOT,
}

for name, src in items_to_copy.items():
    if src.exists():
        dst = BACKUP_DIR / name
        print(f'Copying {src} -> {dst}')
        shutil.copytree(src, dst)
    else:
        print(f'Skipping missing path: {src}')

restore_info = BACKUP_DIR / 'restore_info.txt'
restore_info.write_text(
    f'''Backup time: {datetime.datetime.now():%Y-%m-%d %H:%M:%S}

DATASET_ID={DATASET_ID}
DATASET_NAME={DATASET_NAME}
DATASET_FOLDER={DATASET_FOLDER}
PLANS_NAME={PLANS_NAME}
TRAINER_NAME={TRAINER_NAME}
CONFIGURATION={CONFIGURATION}
TRAIN_FOLDS={TRAIN_FOLDS}

WORKSPACE_BACKUP=brats_workspace_complete
TRAINED_MODEL_BACKUP=trained_model_checkpoints_complete

ORIGINAL_WORKSPACE_DIR={WORKSPACE_DIR}
ORIGINAL_TRAINED_MODEL_ROOT={TRAINED_MODEL_ROOT}
MODEL_BASE={MODEL_BASE}
CHECKPOINT_USED=checkpoint_latest.pth
''',
    encoding='utf-8',
)

print(f'\nFull backup complete: {BACKUP_DIR}')
print('Backed up workspace:', BACKUP_DIR / 'brats_workspace_complete')
print('Backed up trained model:', BACKUP_DIR / 'trained_model_checkpoints_complete')


Copying /content/brats_workspace -> /content/drive/MyDrive/gp2_full_model_backups/Dataset303_ProcessedBraTS128_T1c_Regions_complete_backup_20260425_205631/brats_workspace_complete
Copying /content/drive/MyDrive/gp2/checkpoints/Dataset303_ProcessedBraTS128_T1c_Regions -> /content/drive/MyDrive/gp2_full_model_backups/Dataset303_ProcessedBraTS128_T1c_Regions_complete_backup_20260425_205631/trained_model_checkpoints_complete

Full backup complete: /content/drive/MyDrive/gp2_full_model_backups/Dataset303_ProcessedBraTS128_T1c_Regions_complete_backup_20260425_205631
Backed up workspace: /content/drive/MyDrive/gp2_full_model_backups/Dataset303_ProcessedBraTS128_T1c_Regions_complete_backup_20260425_205631/brats_workspace_complete
Backed up trained model: /content/drive/MyDrive/gp2_full_model_backups/Dataset303_ProcessedBraTS128_T1c_Regions_complete_backup_20260425_205631/trained_model_checkpoints_complete
